## 🧠 AI-Enhanced App Review Analysis & PDF Reporting

This notebook demonstrates a complete pipeline for collecting mobile app user reviews from the Google Play Store, cleaning and analyzing the data, and generating a strategic summary using a large language model (LLM). The output includes a professional, insight-driven report exported in both `.txt` and `.pdf` formats.

---

## 🎯 Goals of this Notebook

* 📥 Fetch and filter user reviews for a specific app from the Google Play Store.
* 🧼 Clean and preprocess text data.
* 🤖 Generate high-level insights and improvement suggestions using LLMs.
* 📄 Format and save the analysis in Markdown, then export it to `.txt` and `.pdf`.

---

## 👥 This Tool Can Be Useful For:

* 🧑‍💼 Product managers & growth strategists
* 📱 App developers & UX designers
* 🧪 User researchers analyzing qualitative feedback
* 📊 Data analysts seeking actionable insights from reviews
* 🎓 Educators teaching text analysis or product strategy

---

## 🔍 How It Works

This notebook follows a step-by-step process:

---

### 📦 Installing Required Libraries

```python
!pip install google-play-scraper
!pip install markdown2 weasyprint openai
```

* `google-play-scraper`: Fetch reviews from the Google Play Store
* `markdown2`: Convert analysis to HTML
* `weasyprint`: Export HTML to PDF
* `openai`: Interact with a large language model (via Hugging Face)

In [ ]:
!pip install google-play-scraper
!pip install markdown2 weasyprint openai

---

### 📂 Importing Necessary Modules

```python
from google_play_scraper import reviews, Sort
from datetime import datetime, timedelta
import re, csv, os
from collections import Counter
from openai import OpenAI
from google.colab import userdata
import markdown
from weasyprint import HTML
```

**Purpose of Each Module:**

| Module                | Purpose                                        |
| --------------------- | ---------------------------------------------- |
| `google_play_scraper` | Collect app reviews from Play Store            |
| `datetime`            | Handle date calculations for filtering reviews |
| `re`                  | Clean and preprocess review text               |
| `csv`                 | Save processed reviews as a CSV file           |
| `OpenAI`              | Connect to LLM (via Hugging Face endpoint)     |
| `markdown`, `HTML`    | Convert LLM response to PDF                    |


In [ ]:
from google_play_scraper import reviews, Sort
from datetime import datetime, timedelta
import re
import csv
from collections import Counter
import os
from openai import OpenAI
from google.colab import userdata
import ast
import markdown
from weasyprint import HTML

---

### 📱 App Configuration

```python
app_package = "com.openai.chatgpt"
```

Specifies which app’s reviews will be fetched.
➡️ In this case: **OpenAI ChatGPT** official mobile app.

---

### 🕓 Targeting Recent Reviews

```python
target_dates = [(datetime.now().date() - timedelta(days=i)) for i in range(2)]
```

Filters for **only the last one days** of user reviews — ensures relevance.

---

### 🧹 Review Preprocessing Function

```python
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
```

Cleans text for analysis:

* Lowercases
* Removes punctuation
* Trims whitespace

---

### 🔁 Review Collection Loop

The loop fetches reviews in batches, cleans them, filters by date, and avoids duplicates.

Key steps:

* Uses pagination (`continuation_token`)
* Filters only reviews from the last one days
* Tracks and avoids repeated content
* Appends cleaned data to `collected_reviews`

```python
while True:
    result, continuation_token = reviews(...)
    ...
```



In [ ]:
app_package = "com.openai.chatgpt"

target_dates = [(datetime.now().date() - timedelta(days=i)) for i in range(2)]

collected_reviews = []
seen_reviews = set()
batch_size = 200
continuation_token = None

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

while True:
    result, continuation_token = reviews(
        app_package,
        lang='en',
        country='us',
        count=batch_size,
        continuation_token=continuation_token,
        sort=Sort.NEWEST
    )
    if not result:
        break

    for r in result:
        review_date = r['at'].date()
        if review_date in target_dates:
            processed = preprocess_text(r['content'])
            if processed and processed not in seen_reviews:
                seen_reviews.add(processed)
                collected_reviews.append({'date': str(review_date), 'processed_content': processed})
        elif review_date < min(target_dates):
            continuation_token = None
            break

    if not continuation_token:
        break

print(f"Number of cleaned reviews: {len(collected_reviews)}")

for r in collected_reviews[:5]:
    print(f"- [{r['date']}] {r['processed_content']}")

Number of cleaned reviews: 1121
- [2025-10-14] my best friendi cant talking with many subject
- [2025-10-14] the best
- [2025-10-14] excellent
- [2025-10-14] nice
- [2025-10-14] op


---
### 📤 Saving Cleaned Reviews to CSV

```python
with open(csv_filename, mode='w', newline='', encoding='utf-8') as csvfile:
    ...
```

All preprocessed and relevant reviews are saved into `cleaned_reviews.csv` for downstream usage or audit.



In [ ]:
csv_filename = "cleaned_reviews.csv"
with open(csv_filename, mode='w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['date', 'processed_content']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for review in collected_reviews:
        writer.writerow(review)


print(f"Reviews have been saved to '{csv_filename}'.")

Reviews have been saved to 'cleaned_reviews.csv'.


---
### 🔐 Retrieving Hugging Face API Token

```python
hf_token = userdata.get('HF_TOKEN')
```

Fetches the API token for interacting with Hugging Face's GPT model.


In [ ]:
hf_token = userdata.get('HF_TOKEN')

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=hf_token,
)

---

### 💬 Prompting the Language Model

A structured **prompt** is prepared for the LLM to generate analysis, asking it to:

* Extract user **insights** (with evidence)
* Provide **strategic product improvement suggestions**
* Maintain **Markdown** formatting
* Avoid sentiment scores or simple keyword grouping

```python
prompt = f"""
You are a senior product analyst...
{sample_texts}
"""
```

LLM is instructed to output two main sections:

1. **Key Insights from User Feedback** (as a table)
2. **Strategic Improvement Suggestions** (as a bullet list)


In [ ]:
sample_texts = "\n".join([f"- {r['processed_content']}" for r in collected_reviews[:10]])

prompt = f"""
You are a senior product analyst and app review specialist.

You are given recent real user reviews of the mobile app: **{app_package}**.

Please review the feedback carefully and generate a professional summary with the following structure:

---

### **Key Insights from User Feedback**

Provide a Markdown table in the following format:

| Insight | How It Appears in the Reviews |
|---------|-------------------------------|

- Extract clear insights based on recurring concerns, questions, expectations, confusion, frustrations, praise, or gaps in user understanding.
- Do NOT focus only on UI or design. Include any topic users mention repeatedly: performance, features, limitations, bugs, pricing, privacy, etc.
- Include relevant user quotes or paraphrased content where helpful.
- Explain briefly why each insight is important or worth noting.

---

### **Strategic Improvement Suggestions**

Based on the patterns in user reviews, infer thoughtful and impactful suggestions to improve the product.

- These suggestions should NOT be just what users explicitly say.
- Think critically: what are the root problems behind the feedback?
- Suggest improvements in:
  - Feature functionality
  - Performance & stability
  - App behavior or reliability
  - Transparency or communication
  - User expectations vs actual experience
  - Onboarding, education, or discoverability
  - Any area of friction or opportunity

- Use bullet points.
- Be specific and actionable.
- Keep it professional and insight-driven.

---

### **Instructions:**

- Do NOT include sentiment analysis.
- Do NOT repeat or list the raw reviews.
- Focus on clarity, structure, and real patterns — not isolated opinions.
- Use Markdown formatting.
- Avoid vague, generic advice.

---

User Reviews:
{sample_texts}
"""

---

### 🤖 Requesting LLM Analysis via Hugging Face

```python
completion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    ...
)
```

Uses **GPT-OSS-20B** (via Hugging Face API) to analyze the reviews and return formatted Markdown content.



In [ ]:
completion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": "You are a professional app review analyst."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.5,
)

explanation = completion.choices[0].message.content

---

### 💾 Saving the LLM Output

```python
with open("review_analysis.txt", "w", encoding="utf-8") as f:
    f.write(explanation)
```

The full report is saved as `review_analysis.txt`.

In [ ]:
with open("review_analysis.txt", "w", encoding="utf-8") as f:
    f.write(explanation)

print("Analysis saved to 'review_analysis.txt'.")

Analysis saved to 'review_analysis.txt'.


---

### 🧾 Converting Markdown to PDF

```python
html_content = markdown.markdown(explanation, extensions=['tables'])
...
HTML(string=html_with_css).write_pdf(pdf_path)
```

* Converts LLM output (Markdown) to styled HTML
* Applies simple CSS for readability
* Exports to `review_analysis.pdf` for sharing or presentation


In [ ]:
html_content = markdown.markdown(explanation, extensions=['tables'])

css = """
<style>
  @page { size: A4; margin: 1cm; }
  body { font-family: Arial, sans-serif; }
</style>
"""

html_with_css = f"<!DOCTYPE html><html><head>{css}</head><body>{html_content}</body></html>"

pdf_path = "review_analysis.pdf"
HTML(string=html_with_css).write_pdf(pdf_path)

print(f"PDF file created: {pdf_path}")

DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.006s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.017s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
INFO:fontTools.subset:kern dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:GPOS dropped
INFO:fontTools.subset:GSUB dropped
DEBUG:f

PDF file created: review_analysis.pdf


---

## ✅ Output Files

| File Name             | Description                            |
| --------------------- | -------------------------------------- |
| `cleaned_reviews.csv` | Raw preprocessed reviews               |
| `review_analysis.txt` | Markdown-formatted LLM output          |
| `review_analysis.pdf` | PDF version of the structured insights |

---

## ✅ Conclusion

Through this notebook, we demonstrated a real-world example of AI-powered product research, extracting actionable insights directly from raw user reviews.

---

## 📌 Key Takeaways

* 🔍 **Play Store reviews** can reveal meaningful trends when filtered and cleaned properly.
* 🤖 **LLMs** like GPT-OSS-20B can summarize unstructured feedback into professional product insights.
* 🧾 Markdown-to-PDF conversion provides a clean format for distribution.
* 🚫 The pipeline avoids sentiment-only summaries, focusing instead on **strategic**, **pattern-based**, and **evidence-backed** suggestions.